<a href="https://colab.research.google.com/github/Laiba-saeed92/Deep-Learning-and-NLP-Projects/blob/main/RAG_application_using_LangChainMistralAI_weviate_db.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install weaviate-client langchain tiktoken pypdf rapidocr-onnxruntime

In [ ]:
#weaviate is a cloud-hosted or self-hosted vector database, you can run it on weaviate cloud or locally using Docker ,kubernetes
WEAVIATE_CLUSTER="https://osnzihxesdadpcfvq1kzca.c0.asia-southeast1.gcp.weaviate.cloud"
WEAVIATE_API_KEY= "a0JCMG9KWGtMeHBBOFBFVl9EbXc0NGRncWsvREhxQ2hPODA5YmVlUnZGRCttVFFWbW55cXpyRGVTNEdJPV92MjAw"


In [ ]:
#Weaviate has become a separate subpackage in langchain so we need to install it first
!pip install langchain-weaviate


In [ ]:
import weaviate
print(weaviate.__version__)


4.18.3


In [ ]:
#from langchain_weaviate.vectorstores import Weaviate-> for old v3 client only
#Correct for Weaviate 4.x:Use WeaviateVectorStore.
from langchain_weaviate import WeaviateVectorStore
import weaviate
from weaviate import WeaviateClient
from weaviate.classes.init import Auth #means you are importing the Auth class from the Weaviate v4 client, and this class is used to handle authentication when connecting to your Weaviate cluster
#Weaviate Cloud requires an API key for access.To connect, you must pass an Auth object, not just a string

In [ ]:
client=weaviate.connect_to_weaviate_cloud( #Instead of manually creating a WeaviateClient and handling authentication, this function does it for you in one step
    cluster_url=WEAVIATE_CLUSTER, #cluster_url is the URL of your Weaviate Cloud instance,This tells the client where your database lives on the internet.
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY), #Authentication is done via Auth.api_key(...) from weaviate.classes.init.
)


In [ ]:
#locale is a Python standard library module that deals with cultural and language settings,
import locale
locale.getprefferedencoding= lambda: "UTF-8" #This function normally returns the default encoding used by your system, e.g., "UTF-8" on most modern systems

In [ ]:
pip install sentence-transformers


In [ ]:
'''from langchain.embeddings import HuggingFaceHubEmbeddings
embedding_model_name ="sentence-transformers/all-mpnet-base-v2"
embeddings= HuggingFaceEmbeddings(model_name= embedding_model_name)
'''


'from langchain.embeddings import HuggingFaceHubEmbeddings\nembedding_model_name ="sentence-transformers/all-mpnet-base-v2"\nembeddings= HuggingFaceEmbeddings(model_name= embedding_model_name)\n'

In [ ]:
from sentence_transformers import SentenceTransformer #Sentence Transformers is a library, not a single model.It provides a framework to generate embeddings for sentences, paragraphs, or documents.
#Internally, it uses Hugging Face Transformer models to compute these embeddings.Example: BERT, RoBERTa, MPNet, etc., can all be used with Sentence Transformers.

# Load the model
embeddings = SentenceTransformer('all-mpnet-base-v2') #all-mpnet-base-v2 is a specific pretrained model available on Hugging Face.It’s part of the Sentence Transformers family and is optimized for sentence-level embeddings.
#all-mpnet-base-v2: a pretrained model you can plug into that engine(sentenceTransformers)Both are available on Hugging Face
#all-mpnet-base-v2 is a pretrained model used within that library to generate embeddings


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
!pip install -U langchain-community pypdf



In [ ]:
from langchain_community.document_loaders import PyPDFLoader
loader= PyPDFLoader("/content/RAG.pdf", extract_images=True)
pages= loader.load()

In [ ]:
!pip install --upgrade --force-reinstall langchain==1.1.2



  Using cached langchain-1.1.2-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_core-1.1.1-py3-none-any.whl.metadata (3.7 kB)
  Using cached langgraph-1.0.4-py3-none-any.whl.metadata (7.8 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langsmith-0.4.56-py3-none-any.whl.metadata (15 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached uuid_utils-0.12.0-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.1 kB)
  Using cached langgraph_checkpoint-3.0.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached langgraph_prebuilt-1.0.5-py3-none-any.whl.metadata (5.2 kB)
  Usi

In [ ]:
import langchain
print(langchain.__version__)  # should print 1.1.2


1.1.2


In [ ]:
#Split text into chunks
import textwrap

chunks = []
for doc in pages:
    raw = textwrap.wrap(doc.page_content, width=1000) # width=1000 splits text into chunks of 1000 characters.
    chunks.extend([raw[0]] + [raw[i-1][-20:] + raw[i] for i in range(1, len(raw))]) #adds 20-character overlap from previous chunk.

print(f"Total chunks: {len(chunks)}")
print(chunks[0][:500]) #now contains all chunks from all pages, ready for embeddings


Total chunks: 121
1 Retrieval-Augmented Generation for Large Language Models: A Survey Yunfan Gaoa, Yun Xiongb, Xinyu Gao b, Kangxiang Jia b, Jinliu Pan b, Yuxi Bic, Yi Dai a, Jiawei Sun a, Meng Wangc, and Haofen Wang a,c aShanghai Research Institute for Intelligent Autonomous Systems, Tongji University bShanghai Key Laboratory of Data Science, School of Computer Science, Fudan University cCollege of Design and Innovation, Tongji University Abstract—Large Language Models (LLMs) showcase impres- sive capabilities 


In [ ]:
from langchain_core.documents import Document
documents = [Document(page_content=chunk) for chunk in chunks] #
# Convert your text chunks into Document objects



In [ ]:
import weaviate
from weaviate.classes.init import Auth
from langchain_community.vectorstores import Weaviate


In [ ]:
# Insert manually
collection = client.collections.get("RAGDocs")  # Or create it first

for doc, vector in zip(chunks, embeddings.embed_documents(chunks)):
    collection.data.insert(properties={"text": doc}, vector=vector)

# Query manually
query_vector = embeddings.embed_query("Your question")
results = collection.query.near_vector(near_vector=query_vector, limit=3)
texts = [obj.properties["text"] for obj in results.objects]


AttributeError: 'SentenceTransformer' object has no attribute 'embed_documents'

In [ ]:
#Storing chunks and embeddings into a vectorstore
vector_db= Weaviate.from_documents(
    documents=documents, embedding=embeddings, client=client,  index_name="MyIndex"

)

AttributeError: 'WeaviateClient' object has no attribute 'schema'